![sandcastle banner image](./sandcastle_intro_banner.png)

In [1]:
# this is the bash command to remove your virtual environment

# sudo /anaconda/bin/conda remove -n py38_selenium --all

In [2]:
# these are the commands to create the coding environment for this notebook

# sudo /anaconda/bin/conda create -y --name py38_selenium python=3.8
# conda activate /anaconda/envs/py38_selenium
# pip install ipykernel selenium webdriver-manager beautifulsoup4 lxml html5lib xlrd wget pandas holidays

In [3]:
# these commands are for installing google-chrome on ubuntu if needed

# wget -nc https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
# sudo apt update
# sudo apt install -f ./google-chrome-stable_current_amd64.deb

In [5]:
# check installed version
!chromium-browser --version

Chromium 135.0.7049.84 snap


check the version based on Release
https://chromedriver.chromium.org/downloads

In [6]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from datetime import datetime
from holidays import Belgium
from time import sleep
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
import math
import requests

### selenium v4

Since version 4 you need to use Chrome Driver Manager and it will automatically  
install the correct Chrome Driver for your environment.

In [7]:
! pip show selenium

Name: selenium
Version: 4.8.3
Summary: 
Home-page: https://www.selenium.dev
Author: 
Author-email: 
License: Apache 2.0
Location: /anaconda/envs/py38_selenium/lib/python3.8/site-packages
Requires: certifi, trio, trio-websocket, urllib3
Required-by: 


In [8]:
print ('Last testrun on: ' + datetime.now().strftime("%d %b %Y"))

Last testrun on: 23 Sep 2025


In [9]:
options = Options()
options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument("--window-size=1920,1200")  # adviced to increase resolution

# here is where the magic happens
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

[WDM] - Downloading: 100%|██████████| 6.83M/6.83M [00:00<00:00, 112MB/s]


In [10]:
# open the webpage where we can find all tides
driver.get("https://odnature.naturalsciences.be/marine-forecasting-centre/nl/harmonic-tides")

In [11]:
# maximize the window
driver.maximize_window()

In [12]:
# check the image (this is used for debugging code)
driver.save_screenshot('screen.png')

True

## change the start date

In [13]:
# Change the start date formaat JJJJ-MM-DD
begindatum = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.CSS_SELECTOR, "input[id='harmonic-start-date']")))

begindatum.clear() # Clear any existing value
begindatum.send_keys("2026-01-01") # CHANGE THE YEAR

## change the end date

In [14]:
# Locate the end date input field
einddatum = WebDriverWait(driver, 10).until(EC.visibility_of_element_located((By.CSS_SELECTOR, "input[id='harmonic-end-date']")))

# Click the input field to make it editable
einddatum.click()

einddatum.clear() # Clear any existing value
einddatum.send_keys("2026-12-31") # CHANGE THE YEAR

### downloading website table

In [15]:
button = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.ID, "update-table")))
button.click()

In [16]:
# wait 5 seconds, for the webpage to load
sleep(5)

In [17]:
# Get the page source after all interactions
page_source = driver.page_source

In [18]:
# Parse the page source with BeautifulSoup
soup = BeautifulSoup(page_source, 'html.parser')

In [19]:
# Extract the table with the specific attribute name="tableResults"
table = soup.find('table', {'name': 'tableResults'})

In [20]:
# Convert the table to a DataFrame
df = pd.read_html(str(table))[0]

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 731 entries, 0 to 730
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Datum               730 non-null    object
 1   Hoog water [m TAW]  730 non-null    object
 2   Tijd [UTC]          730 non-null    object
 3   Laag water [m TAW]  730 non-null    object
 4   Tijd [UTC].1        730 non-null    object
dtypes: object(5)
memory usage: 28.7+ KB


In [22]:
df

,Datum,Hoog water [m TAW],Tijd [UTC],Laag water [m TAW],Tijd [UTC].1
0,NaN,NaN,NaN,NaN,NaN
1,2026-01-01,4.26,09:50,0.81,04:06
2,2026-01-01,4.22,22:28,0.55,16:33
3,2026-01-02,4.43,10:50,0.61,05:11
4,2026-01-02,4.36,23:24,0.47,17:33
...,...,...,...,...,...
726,2026-12-29,4.48,16:08,0.72,22:38
727,2026-12-30,4.27,04:30,0.27,11:10
728,2026-12-30,4.29,17:06,0.87,23:30
729,2026-12-31,4.14,05:28,--.--,--.--


### clean and prepare the data

In [23]:
# change the column names
df.columns = ['datum', 'hoog_water', 'hoog_tijd', 'laag_water', 'laag_tijd']

In [24]:
# remove record 0 because it is all NaN
df = df.drop(0, axis=0)

In [25]:
# first replace the values "--.--"" into NaN
x = {"--.--":np.nan}
df = df.replace(x)

# change hoog en laag water from string into float numbers
df = df.astype({'hoog_water':'float', 'laag_water':'float'})

In [26]:
# Convert 'datum' to datetime format
df['datum'] = pd.to_datetime(df['datum'])

In [27]:
# Combine the date from 'datum' with the time values from 'hoog_tijd' and 'laag_tijd'
df['hoog_tijd'] = pd.to_datetime(df['datum'].dt.strftime('%Y-%m-%d') + ' ' + df['hoog_tijd'])
df['laag_tijd'] = pd.to_datetime(df['datum'].dt.strftime('%Y-%m-%d') + ' ' + df['laag_tijd'])

In [28]:
# Localize to UTC
df['hoog_tijd'] = df['hoog_tijd'].dt.tz_localize('UTC')
df['laag_tijd'] = df['laag_tijd'].dt.tz_localize('UTC')

In [29]:
# Converteer de tijden naar de lokale tijdzone van Oostende, België ("Europe/Brussels")
df['laag_tijd'] = df['laag_tijd'].dt.tz_convert('Europe/Brussels')
df['hoog_tijd'] = df['hoog_tijd'].dt.tz_convert('Europe/Brussels')

In [30]:
# remove the column "datum"
df.drop(["datum"], axis=1, inplace=True)

In [31]:
# Get basic statistics for each column
print(df.describe())

       hoog_water  laag_water
count  705.000000  705.000000
mean     4.290383    0.519759
std      0.405535    0.381216
min      3.200000   -0.310000
25%      4.010000    0.250000
50%      4.310000    0.490000
75%      4.620000    0.790000
max      5.080000    1.500000


In [32]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 730 entries, 1 to 730
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype                          
---  ------      --------------  -----                          
 0   hoog_water  705 non-null    float64                        
 1   hoog_tijd   705 non-null    datetime64[ns, Europe/Brussels]
 2   laag_water  705 non-null    float64                        
 3   laag_tijd   705 non-null    datetime64[ns, Europe/Brussels]
dtypes: datetime64[ns, Europe/Brussels](2), float64(2)
memory usage: 22.9 KB


In [33]:
# backup/store the data into a CSV file
df.to_csv('getijden.csv', index=0)

## we now need to filter for dates that allow to build the castle during low tide

In [34]:
# Filter the data to get rows where low tide is between 11:00 and 13:00
# in past experience we saw when predicted low tide ~12u20 CEST means water reach castle ~16u00 4/okt/2015 (event 10:00 tot 18:00)
# in the past we saw when predicted low tide ~11u56 CEST means water reach castle ~16u00 3/sep/2011
low_tide_filter = (df['laag_tijd'].dt.hour >= 11) & (df['laag_tijd'].dt.hour < 13)

In [35]:
# Filter voor maanden tussen mei en oktober (maanden 5 t/m 10) 
# maar negeer de maanden juli (7) en augustus (8)
month_filter = (df['laag_tijd'].dt.month >= 5) & (df['laag_tijd'].dt.month <= 10) & ~(df['laag_tijd'].dt.month.isin([7, 8]))

In [36]:
# Drop NaN values from 'laag_tijd' and get unique years
unique_years = df['laag_tijd'].dropna().dt.year.unique().tolist()

In [37]:
# Fetch Belgian holidays for the unique years
belgian_holidays = Belgium(years=unique_years)

In [38]:
belgian_holidays

{datetime.date(2026, 1, 1): 'Nieuwjaar', datetime.date(2026, 4, 5): 'Pasen', datetime.date(2026, 4, 6): 'Paasmaandag', datetime.date(2026, 5, 1): 'Dag van de Arbeid', datetime.date(2026, 5, 14): 'O. L. H. Hemelvaart', datetime.date(2026, 5, 24): 'Pinksteren', datetime.date(2026, 5, 25): 'Pinkstermaandag', datetime.date(2026, 7, 21): 'Nationale feestdag', datetime.date(2026, 8, 15): 'O. L. V. Hemelvaart', datetime.date(2026, 11, 1): 'Allerheiligen', datetime.date(2026, 11, 11): 'Wapenstilstand', datetime.date(2026, 12, 25): 'Kerstmis'}

In [39]:
# Convert the keys (dates) of the belgian_holidays dictionary to a list
feestdagen = [str(date) for date in belgian_holidays.keys()]

In [40]:
feestdag_filter = df['laag_tijd'].dt.date.astype(str).isin(feestdagen)

In [41]:
# Filter voor enkel weekenddagen: zaterdag (5) of zondag (6)
weekend_filter = df['laag_tijd'].dt.weekday.isin([5, 6])

In [42]:
# Combineer de filters met een OR-operatie
weekend_feestdag = weekend_filter | feestdag_filter

In [43]:
# Combine all the filters
combined_filter = low_tide_filter & month_filter & weekend_feestdag

In [44]:
# Get the days that satisfy all conditions = best days
df[combined_filter]

,hoog_water,hoog_tijd,laag_water,laag_tijd
257,3.77,2026-05-09 06:31:00+02:00,1.12,2026-05-09 12:50:00+02:00
313,4.12,2026-06-06 05:24:00+02:00,0.85,2026-06-06 11:50:00+02:00
315,4.03,2026-06-07 06:17:00+02:00,0.95,2026-06-07 12:44:00+02:00
341,4.51,2026-06-20 05:45:00+02:00,0.56,2026-06-20 12:18:00+02:00
551,4.19,2026-10-03 06:06:00+02:00,0.72,2026-10-03 12:45:00+02:00
579,4.05,2026-10-17 05:25:00+02:00,0.85,2026-10-17 11:45:00+02:00
581,3.76,2026-10-18 06:02:00+02:00,1.12,2026-10-18 12:28:00+02:00


In [45]:
max_length = max([len(row['hoog_tijd'].strftime("%A %d %B %Y")) for _, row in df[combined_filter].iterrows()])

for index, row in df[combined_filter].iterrows():
    date_str = row['hoog_tijd'].strftime("%A %d %B %Y")
    time_str = row['laag_tijd'].strftime("%Hu%M")
    print(f"{date_str:<{max_length}} (met laag water om {time_str})")

Saturday 09 May 2026     (met laag water om 12u50)
Saturday 06 June 2026    (met laag water om 11u50)
Sunday 07 June 2026      (met laag water om 12u44)
Saturday 20 June 2026    (met laag water om 12u18)
Saturday 03 October 2026 (met laag water om 12u45)
Saturday 17 October 2026 (met laag water om 11u45)
Sunday 18 October 2026   (met laag water om 12u28)
